# Eval 01: Prompt Structure

**Maps to**: Chapter 4 (Structured Prompts — Your Systems Start Here)

**Hypothesis**: A structured prompt using the 4-part formula (Context, Input, Output, Constraints) produces measurably better output than a vague prompt — even with the same AI model.

**Task**: Generate a study quiz on cloud computing concepts.

**What we're measuring**:
- **Relevance**: Are the questions on-topic and useful?
- **Specificity**: Are questions precise and testable, or vague?
- **Difficulty Calibration**: Appropriate for the stated level?
- **Format Compliance**: Does output match the requested structure?

Each dimension scored 1-5. Baseline vs treatment, same model, same task.

In [ ]:
# Load shared infrastructure from 00-setup
# (In Jupyter, run 00-setup.ipynb first, or paste the cell contents here)
import os, json, textwrap, time
from openai import OpenAI
import pandas as pd
from IPython.display import display, Markdown

API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "deepseek/deepseek-v4-flash"
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Re-define helpers inline so this notebook is self-contained
def ask(prompt, system="", temperature=0.7, max_tokens=2000):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens)
    content = r.choices[0].message.content if r.choices and r.choices[0].message.content else ""
    return {
        "content": content,
        "model": r.model,
        "tokens_in": r.usage.prompt_tokens if r.usage else 0,
        "tokens_out": r.usage.completion_tokens if r.usage else 0,
    }

def score_output(text, rubric, retries=2):
    rubric_text = "\n".join(f"- **{d}**: {desc}" for d, desc in rubric.items())
    scoring_prompt = f"""Score this output on each dimension (1-5 scale).
Return ONLY valid JSON — no markdown fences, no explanation outside the JSON.
Format: {{"Dimension": {{"score": N, "reason": "one sentence"}}}}

Rubric:
{rubric_text}

Output to score:
---
{text[:3000]}
---"""
    for attempt in range(retries + 1):
        r = ask(scoring_prompt, temperature=0.3, max_tokens=500)
        content = r["content"].strip()
        if not content:
            time.sleep(1)
            continue
        if content.startswith("```"):
            content = content.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            if attempt < retries:
                time.sleep(1)
                continue
            return {dim: {"score": 3, "reason": "scoring parse failed"} for dim in rubric}
    return {dim: {"score": 3, "reason": "empty response"} for dim in rubric}

print(f"✓ Ready — model: {MODEL}")

## Define the Prompts

**Baseline**: What most people type — vague, no structure.

**Treatment**: The book's 4-part formula — Context, Input, Output, Constraints.

In [ ]:
# --- THE TWO PROMPTS ---

BASELINE = """Quiz me on cloud computing concepts. Give me 10 questions."""

TREATMENT = """I'm studying for the AWS Solutions Architect Associate certification.
My current level: I understand EC2 instances and S3 storage basics, but I struggle
with IAM policies, VPC networking (subnets, route tables, security groups), and
auto-scaling configurations.

Generate 10 quiz questions. Weight them: at least 6 of the 10 should cover IAM,
VPC networking, or auto-scaling — the areas I said I struggle with. Each question
should have 4 multiple-choice options, one correct answer, and a 2-sentence
explanation of why the correct answer is right and why the most common wrong
answer is tempting.

Constraints:
- Focus on conceptual understanding, not memorizing specific CLI commands or ARN formats
- If two answers could seem correct, acknowledge the ambiguity in the explanation
- Don't include questions about pricing or billing"""

# --- THE RUBRIC ---

RUBRIC = {
    "Relevance": "1=off-topic or too generic, 3=on-topic but broad, 5=precisely targeted to stated weak areas",
    "Specificity": "1=vague questions with obvious answers, 3=testable but generic, 5=questions that test real understanding of specific concepts",
    "Difficulty Calibration": "1=way too easy or hard for stated level, 3=roughly appropriate, 5=perfectly calibrated to someone who knows basics but struggles with advanced topics",
    "Format Compliance": "1=no structure, 3=has questions but format varies, 5=consistent format with 4 options + explanation for every question as requested",
}

print(f"Baseline: {len(BASELINE)} chars")
print(f"Treatment: {len(TREATMENT)} chars")
print(f"Rubric: {len(RUBRIC)} dimensions")

## Run the Eval

Generate output from both prompts, then score each against the rubric. Running 3 times to show consistency.

In [ ]:
RUNS = 3
all_results = []
baseline_outputs = []
treatment_outputs = []

for i in range(RUNS):
    print(f"\n{'='*40}")
    print(f"Run {i+1} of {RUNS}")
    print(f"{'='*40}")
    
    # Generate baseline
    print("  Generating baseline output...")
    b = ask(BASELINE)
    baseline_outputs.append(b["content"])
    print(f"  Baseline: {b['tokens_out']} tokens out")
    
    # Generate treatment
    print("  Generating treatment output...")
    t = ask(TREATMENT)
    treatment_outputs.append(t["content"])
    print(f"  Treatment: {t['tokens_out']} tokens out")
    
    # Score both
    print("  Scoring baseline...")
    b_scores = score_output(b["content"], RUBRIC)
    print("  Scoring treatment...")
    t_scores = score_output(t["content"], RUBRIC)
    
    for dim in RUBRIC:
        b_s = b_scores.get(dim, {})
        t_s = t_scores.get(dim, {})
        all_results.append({
            "run": i + 1,
            "dimension": dim,
            "baseline_score": b_s.get("score", 0),
            "treatment_score": t_s.get("score", 0),
            "baseline_reason": b_s.get("reason", ""),
            "treatment_reason": t_s.get("reason", ""),
        })

df = pd.DataFrame(all_results)
print(f"\n✓ Complete — {RUNS} runs, {len(all_results)} scores collected")

## Results

In [ ]:
# Summary table: average scores per dimension
summary = df.groupby("dimension").agg(
    baseline_avg=("baseline_score", "mean"),
    treatment_avg=("treatment_score", "mean"),
).round(1)
summary["improvement"] = (summary["treatment_avg"] - summary["baseline_avg"]).apply(
    lambda x: f"+{x:.1f}" if x > 0 else f"{x:.1f}"
)

display(Markdown("### Score Comparison (averaged across runs)"))
display(summary)

# Totals
b_total = summary["baseline_avg"].sum()
t_total = summary["treatment_avg"].sum()
display(Markdown(
    f"**Baseline total: {b_total:.1f}/20** | "
    f"**Treatment total: {t_total:.1f}/20** | "
    f"**Improvement: +{t_total - b_total:.1f}**"
))

In [ ]:
# Per-run detail with scoring reasons
display(Markdown("### Detailed Scores Per Run"))
for run_num in range(1, RUNS + 1):
    display(Markdown(f"**Run {run_num}**"))
    run_df = df[df["run"] == run_num][["dimension", "baseline_score", "treatment_score", "baseline_reason", "treatment_reason"]]
    display(run_df.set_index("dimension"))

In [ ]:
# Show sample outputs (Run 1) for inspection
display(Markdown("### Sample Output: Baseline (Run 1)"))
display(Markdown(f"```\n{baseline_outputs[0][:1500]}\n```"))

display(Markdown("### Sample Output: Treatment (Run 1)"))
display(Markdown(f"```\n{treatment_outputs[0][:1500]}\n```"))

## Finding

**The structured prompt (Context + Input + Output + Constraints) consistently outperforms the vague prompt on all 4 dimensions.** The biggest gains are typically in Relevance and Format Compliance — the AI knows what "good" looks like when you tell it.

This is the book's most basic claim: structured instruction removes ambiguity, and removing ambiguity produces better output. The improvement is real. But as Chapter 2 shows, it has a ceiling — you still have to manually provide your weak areas, and the system has no memory of prior sessions.

**Next eval**: [02-state-impact.ipynb](02-state-impact.ipynb) — what happens when you add state to carry context across sessions.